In [ ]:
#for getting highest cnn score pose of each compound
# conda activate gnina-env
#export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH
#gnina   -r Protein.pdb   -l all_ligands.sdf   --center_x 14.265 --center_y 25.081 --center_z 21.716   --size_x   50 --size_y   40 --size_z   42   --exhaustiveness 64   -o batch_docked.sdf.gz --device 0

from rdkit import Chem
import pandas as pd

input_sdf = "batch_docked_ttbk1.sdf"
output_csv = "ttbk1_top_pose_highest_cnn_affinity.csv"

supplier = Chem.SDMolSupplier(input_sdf)
pose_data = []

for mol in supplier:
    if mol is None:
        continue
    try:
        identifier = mol.GetProp("IDENTIFIER")
        cnn_affinity = float(mol.GetProp("CNNaffinity"))
        minimized_affinity = float(mol.GetProp("minimizedAffinity"))
        cnn_score = float(mol.GetProp("CNNscore"))
        pose_data.append({
            "IDENTIFIER": identifier,
            "CNNaffinity": cnn_affinity,
            "minimizedAffinity": minimized_affinity,
            "CNNscore": cnn_score
        })
    except Exception as e:
        print(f"Skipping molecule due to error: {e}")

# Convert to DataFrame
df = pd.DataFrame(pose_data)

# Sort by CNNaffinity within each identifier group (highest first)
df_sorted = df.sort_values(by=['IDENTIFIER', 'CNNaffinity'], ascending=[True, False])

# Keep only the pose with the highest CNNaffinity per identifier
top_poses = df_sorted.groupby('IDENTIFIER').first().reset_index()

# Rank all compounds by CNNaffinity (highest first)
top_poses = top_poses.sort_values(by='CNNaffinity', ascending=False)
top_poses['Rank'] = range(1, len(top_poses) + 1)

# Save to CSV
top_poses.to_csv(output_csv, index=False)

print(f"Top poses with highest CNN affinity saved and ranked to {output_csv}")



# Example usage
# extract_gnina_bestpose_to_csv('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/gnina', 'gnina_abl1_bestposes.csv')


Top poses with highest CNN affinity saved and ranked to ttbk1_top_pose_highest_cnn_affinity.csv


In [25]:
##for saving best poses in 3d sdf
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/gnina')
from rdkit import Chem
import os

input_sdf = "batch_docked_dyrk1a.sdf"
output_dir = "best_pose_sdfs_dyrk1a"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Dictionary to store best pose per identifier
best_poses = {}

# Read all molecules from SDF
supplier = Chem.SDMolSupplier(input_sdf)
for mol in supplier:
    if mol is None:
        continue
    try:
        identifier = mol.GetProp("IDENTIFIER")
        cnn_affinity = float(mol.GetProp("CNNaffinity"))
        # Store the best (highest CNNaffinity) pose
        if (identifier not in best_poses) or (cnn_affinity > best_poses[identifier][1]):
            best_poses[identifier] = (mol, cnn_affinity)
    except Exception as e:
        print(f"Skipping molecule due to error: {e}")

# Write each best pose to its own SDF file
for identifier, (mol, cnn_affinity) in best_poses.items():
    # Clean identifier for safe filename
    safe_identifier = "".join([c if c.isalnum() or c in "-_." else "_" for c in identifier])
    out_path = os.path.join(output_dir, f"{safe_identifier}.sdf")
    writer = Chem.SDWriter(out_path)
    writer.write(mol)
    writer.close()

print(f"Saved best pose SDFs for {len(best_poses)} compounds in '{output_dir}' directory.")


Saved best pose SDFs for 34 compounds in 'best_pose_sdfs_dyrk1a' directory.
